In [3]:
import os, re
import pandas as pd
import ast
import json
import openpyxl

In [3]:
# go one directory up from notebooks/ into data/
DATA_DIR = os.path.join("..", "data")
META_PATH = os.path.join(DATA_DIR, "metadata.csv")

meta = pd.read_csv(META_PATH)
meta.head()

,id,title,author,authoryearofbirth,authoryearofdeath,language,downloads,subjects,type
0,PG10000,The Magna Carta,Anonymous,NaN,NaN,['en'],1083,"{'Magna Carta', 'Constitutional history -- Eng...",Text
1,PG10001,Apocolocyntosis,"Seneca, Lucius Annaeus",NaN,65.0,['en'],2048,"{'Claudius, Emperor of Rome, 10 B.C.-54 A.D. -...",Text
2,PG10002,The House on the Borderland,"Hodgson, William Hope",1877.0,1918.0,['en'],1752,{'Science fiction'},Text
3,PG10003,"My First Years as a Frenchwoman, 1876-1879","Waddington, Mary King",1833.0,1923.0,['en'],260,"{'France -- History -- Third Republic, 1870-19...",Text
4,PG10004,The Warriors,"Lindsay, Anna Robertson Brown",1864.0,1948.0,['en'],205,{'Christianity'},Text


In [4]:
meta.columns.tolist()

['id',
 'title',
 'author',
 'authoryearofbirth',
 'authoryearofdeath',
 'language',
 'downloads',
 'subjects',
 'type']

In [5]:
n_texts = len(os.listdir(os.path.join("../data", "text")))
print(n_texts, "cleaned text files")

48709 cleaned text files


In [6]:
meta[meta["title"].str.contains("Pride and Prejudice", case=False, na=False)]

,id,title,author,authoryearofbirth,authoryearofdeath,language,downloads,subjects,type
3806,PG1342,Pride and Prejudice,"Austen, Jane",1775.0,1817.0,['en'],79774,"{'Sisters -- Fiction', 'Social classes -- Fict...",Text
11841,PG20686,Pride and Prejudice,"Austen, Jane",1775.0,1817.0,['en'],1164,"{'Sisters -- Fiction', 'Social classes -- Fict...",Sound
11842,PG20687,Pride and Prejudice,"Austen, Jane",1775.0,1817.0,['en'],1630,"{'Sisters -- Fiction', 'Social classes -- Fict...",Sound
18075,PG26301,Pride and Prejudice,"Austen, Jane",1775.0,1817.0,['en'],1698,"{'Sisters -- Fiction', 'Social classes -- Fict...",Sound
30434,PG37431,"Pride and Prejudice, a play founded on Jane Au...","MacKaye, Steele, Mrs.",1845.0,1924.0,['en'],2440,"{'Social classes -- Drama', 'Sisters -- Drama'...",Text
36255,PG42671,Pride and Prejudice,"Austen, Jane",1775.0,1817.0,['en'],3646,"{'Sisters -- Fiction', 'Social classes -- Fict...",Text


In [7]:
en_meta = meta[meta["language"] == "['en']"]
en_meta

,id,title,author,authoryearofbirth,authoryearofdeath,language,downloads,subjects,type
0,PG10000,The Magna Carta,Anonymous,NaN,NaN,['en'],1083,"{'Magna Carta', 'Constitutional history -- Eng...",Text
1,PG10001,Apocolocyntosis,"Seneca, Lucius Annaeus",NaN,65.0,['en'],2048,"{'Claudius, Emperor of Rome, 10 B.C.-54 A.D. -...",Text
2,PG10002,The House on the Borderland,"Hodgson, William Hope",1877.0,1918.0,['en'],1752,{'Science fiction'},Text
3,PG10003,"My First Years as a Frenchwoman, 1876-1879","Waddington, Mary King",1833.0,1923.0,['en'],260,"{'France -- History -- Third Republic, 1870-19...",Text
4,PG10004,The Warriors,"Lindsay, Anna Robertson Brown",1864.0,1948.0,['en'],205,{'Christianity'},Text
...,...,...,...,...,...,...,...,...,...
76796,PG9997,"France and England in North America, Part III:...","Parkman, Francis",1823.0,1893.0,['en'],233,{'Canada -- History -- To 1763 (New France)'},Text
76797,PG9998,Poems,"Betham, Matilda",1776.0,1852.0,['en'],162,{'Poetry'},Text
76798,PG9999,"Harriet, the Moses of Her People","Bradford, Sarah H. (Sarah Hopkins)",1818.0,1912.0,['en'],1114,"{'African Americans -- Biography', 'Enslaved p...",Text
76800,PG99,Collected Articles of Frederick Douglass,"Douglass, Frederick",1818.0,1895.0,['en'],488,"{'Reconstruction (U.S. history, 1865-1877)', '...",Text


In [8]:
en_meta['subjects'].value_counts()

subjects
{'Fiction'}                                                                                                                                                                                                                961
{'English wit and humor -- Periodicals'}                                                                                                                                                                                   551
{'Poetry'}                                                                                                                                                                                                                 363
{'Short stories', 'Science fiction'}                                                                                                                                                                                       357
{'Science fiction'}                                                                                

In [9]:
def parse_subjects(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) and x.strip() else set()
    except Exception:
        return set()

meta["subjects_set"] = meta["subjects"].apply(parse_subjects)

In [10]:
meta.head()

,id,title,author,authoryearofbirth,authoryearofdeath,language,downloads,subjects,type,subjects_set
0,PG10000,The Magna Carta,Anonymous,NaN,NaN,['en'],1083,"{'Magna Carta', 'Constitutional history -- Eng...",Text,"{Magna Carta, Constitutional history -- Englan..."
1,PG10001,Apocolocyntosis,"Seneca, Lucius Annaeus",NaN,65.0,['en'],2048,"{'Claudius, Emperor of Rome, 10 B.C.-54 A.D. -...",Text,"{Claudius, Emperor of Rome, 10 B.C.-54 A.D. --..."
2,PG10002,The House on the Borderland,"Hodgson, William Hope",1877.0,1918.0,['en'],1752,{'Science fiction'},Text,{Science fiction}
3,PG10003,"My First Years as a Frenchwoman, 1876-1879","Waddington, Mary King",1833.0,1923.0,['en'],260,"{'France -- History -- Third Republic, 1870-19...",Text,"{France -- Social life and customs, France -- ..."
4,PG10004,The Warriors,"Lindsay, Anna Robertson Brown",1864.0,1948.0,['en'],205,{'Christianity'},Text,{Christianity}


In [11]:
def parse_subjects(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) and x.strip() else set()
    except Exception:
        return set()

meta["subjects_set"] = meta["subjects"].apply(parse_subjects)

In [12]:
fiction_meta = meta[meta["subjects_set"].apply(
    lambda s: any("fiction" in subj.lower() for subj in s)
)]

In [13]:
fiction_meta[fiction_meta["author"] == "Austen, Jane"]['title'].value_counts()

title
Pride and Prejudice                                                                                          5
Persuasion                                                                                                   3
Sense and Sensibility                                                                                        3
Northanger Abbey                                                                                             2
Mansfield Park                                                                                               2
Emma                                                                                                         2
Lady Susan                                                                                                   2
Love and Freindship [sic]                                                                                    2
Raison et sensibilité, ou les deux manières d'aimer (Tome 3)                                              

In [15]:
#find ID for pride and prejudice
book = meta[meta["title"].str.contains("Pride and Prejudice", case=False, na=False)].iloc[0]
book_id = book["id"]
book[["id", "title", "author", "language"]]

id                       PG1342
title       Pride and Prejudice
author             Austen, Jane
language                 ['en']
Name: 3806, dtype: object

In [16]:
def load_text(book_id, base_dir="../data/text"):
    path = os.path.join(base_dir, f'PG{book_id}_text.txt')
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return f.read()
    raise FileNotFoundError(f"No text found for PG{book_id}")

In [17]:
text = load_text(1342)
print(text[:1000])  # print the first 500 characters





                            [Illustration:

                             GEORGE ALLEN
                               PUBLISHER

                        156 CHARING CROSS ROAD
                                LONDON

                             RUSKIN HOUSE
                                   ]

                            [Illustration:

               _Reading Jane’s Letters._      _Chap 34._
                                   ]




                                PRIDE.
                                  and
                               PREJUDICE

                                  by
                             Jane Austen,

                           with a Preface by
                           George Saintsbury
                                  and
                           Illustrations by
                             Hugh Thomson

                         [Illustration: 1894]

                       Ruskin       156. Charing
                       House.        Cross Road.


In [18]:
CHAPTER_PATTERNS = [
    r"(?m)^\s*(?:CHAPTER|Chapter|chapter)\s+[IVXLCDM]+[\s\.\-\—:]*",   # Roman numerals
    r"(?m)^\s*(?:CHAPTER|Chapter|chapter)\s+\d+[\s\.\-\—:]*",          # Arabic numerals
    r"(?m)^\s*(?:BOOK|Book|Part|PART)\s+[IVXLCDM\d]+[\s\.\-\—:]*",     # Book/Part divisions
]

def detect_chapter_headings(text):
    matches = []
    for pat in CHAPTER_PATTERNS:
        matches += [m.start() for m in re.finditer(pat, text)]
    matches = sorted(set(matches))
    return matches

def split_into_chapters(text, min_chars=500):
    indices = detect_chapter_headings(text)
    if not indices:
        return [text]
    chunks = []
    for i, start in enumerate(indices):
        end = indices[i + 1] if i + 1 < len(indices) else len(text)
        chapter = text[start:end].strip()
        if len(chapter) >= min_chars:
            chunks.append(chapter)
    return chunks

In [19]:
chapters = split_into_chapters(text)
print(f"Detected {len(chapters)} chapters.")

for i, ch in enumerate(chapters[:5], 1):
    print(f"\n---- Chapter {i} ----\n")
    print(ch[:400])

Detected 61 chapters.

---- Chapter 1 ----

Chapter I.]


It is a truth universally acknowledged, that a single man in possession
of a good fortune must be in want of a wife.

However little known the feelings or views of such a man may be on his
first entering a neighbourhood, this truth is so well fixed in the minds
of the surrounding families, that he is considered as the rightful
property of some one or other of their daughters.

“My de

---- Chapter 2 ----

CHAPTER II.


[Illustration]

Mr. Bennet was among the earliest of those who waited on Mr. Bingley. He
had always intended to visit him, though to the last always assuring his
wife that he should not go; and till the evening after the visit was
paid she had no knowledge of it. It was then disclosed in the following
manner. Observing his second daughter employed in trimming a hat, he
suddenly addre

---- Chapter 3 ----

CHAPTER III.


[Illustration]

Not all that Mrs. Bennet, however, with the assistance of her five
daughters, c

In [68]:
os.makedirs("../data/chapters", exist_ok=True)

In [69]:
for i in range(10):
    print(chapters[i][:200])

Chapter I.]


It is a truth universally acknowledged, that a single man in possession
of a good fortune must be in want of a wife.

However little known the feelings or views of such a man may be on h
CHAPTER II.


[Illustration]

Mr. Bennet was among the earliest of those who waited on Mr. Bingley. He
had always intended to visit him, though to the last always assuring his
wife that he should not 
CHAPTER III.


[Illustration]

Not all that Mrs. Bennet, however, with the assistance of her five
daughters, could ask on the subject, was sufficient to draw from her
husband any satisfactory descript
CHAPTER IV.


[Illustration]

When Jane and Elizabeth were alone, the former, who had been cautious in
her praise of Mr. Bingley before, expressed to her sister how very much
she admired him.

“He is 
CHAPTER V.


[Illustration]

Within a short walk of Longbourn lived a family with whom the Bennets
were particularly intimate. Sir William Lucas had been formerly in trade
in Meryton, where he had

In [70]:
def clean_gutenberg(text):
    # remove illustration or copyright spans
    text = re.sub(
        r"\[[_\s]*Illustration[:\s]*.*?\]",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    text = re.sub(
        r"\[_Copyright.*?_\]",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    text = re.sub(
        r"\[\[.*?\]\]",
        "",
        text,
        flags=re.DOTALL
    )
    # collapse multiple newlines
    text = re.sub(r"\n{2,}", "\n\n", text)
    # trim leading/trailing whitespace
    return text.strip()

clean_chapters = [clean_gutenberg(ch) for ch in chapters]
for i in range(10):
    print(clean_chapters[i][:200])

Chapter I.]

It is a truth universally acknowledged, that a single man in possession
of a good fortune must be in want of a wife.

However little known the feelings or views of such a man may be on hi
CHAPTER II.

Mr. Bennet was among the earliest of those who waited on Mr. Bingley. He
had always intended to visit him, though to the last always assuring his
wife that he should not go; and till the 
CHAPTER III.

Not all that Mrs. Bennet, however, with the assistance of her five
daughters, could ask on the subject, was sufficient to draw from her
husband any satisfactory description of Mr. Bingle
CHAPTER IV.

When Jane and Elizabeth were alone, the former, who had been cautious in
her praise of Mr. Bingley before, expressed to her sister how very much
she admired him.

“He is just what a young
CHAPTER V.

Within a short walk of Longbourn lived a family with whom the Bennets
were particularly intimate. Sir William Lucas had been formerly in trade
in Meryton, where he had made a tolerable

In [71]:
def strip_chapter_headers(ch_text: str) -> str:
    """
    Removes all content before the first double newline.
    Useful when each chapter header is followed by one blank line
    and then the first paragraph of text.
    """
    # normalize newlines (some files use \r\n)
    ch_text = ch_text.replace("\r\n", "\n")

    # find the first paragraph break
    parts = ch_text.split("\n\n", 1)

    # if we found one, take everything after it; else return the text as-is
    if len(parts) > 1:
        cleaned = parts[1]
    else:
        cleaned = ch_text

    # collapse extra blank lines and trim
    cleaned = re.sub(r"\n{2,}", "\n\n", cleaned).strip()
    return cleaned

clean_chapters = [strip_chapter_headers(ch) for ch in clean_chapters]

In [77]:
print(clean_chapters[10])

When the ladies removed after dinner Elizabeth ran up to her sister, and
seeing her well guarded from cold, attended her into the drawing-room,
where she was welcomed by her two friends with many professions of
pleasure; and Elizabeth had never seen them so agreeable as they were
during the hour which passed before the gentlemen appeared. Their powers
of conversation were considerable. They could describe an entertainment
with accuracy, relate an anecdote with humour, and laugh at their
acquaintance with spirit.

But when the gentlemen entered, Jane was no longer the first object;
Miss Bingley’s eyes were instantly turned towards Darcy, and she had
something to say to him before he had advanced many steps. He addressed
himself directly to Miss Bennet with a polite congratulation; Mr. Hurst
also made her a slight bow, and said he was “very glad;” but diffuseness
and warmth remained for Bingley’s salutation. He was full of joy and
attention. The first half hour was spent in piling up the

In [78]:
for ch in clean_chapters:
    save_path = os.path.join("../data/chapters", f"pride_and_prejudice_{clean_chapters.index(ch)+1}.txt")
    with open(save_path, "w", encoding="utf-8") as f:
        f.write(ch)

In [3]:
import pandas as pd
gold = pd.read_excel("../data/manual/pride_and_prejudice.xlsx", sheet_name="ALL INSTANCES")
gold.head(10)

,Volume,Chapter,Graph Chapter,Character,N,DC,C,I,DN,A,B,FID,Score
0,1.0,7,7,Captain Carter,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,4
1,1.0,9,9,Captain Carter,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2
2,1.0,5,5,Charlotte Lucas,3.0,2.0,5.0,2.0,1.0,0.0,0.0,0.0,13
3,1.0,6,6,Charlotte Lucas,7.0,3.0,5.0,2.0,0.0,1.0,0.0,0.0,18
4,1.0,9,9,Charlotte Lucas,3.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,6
5,1.0,13,13,Charlotte Lucas,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2
6,1.0,18,18,Charlotte Lucas,5.0,1.0,3.0,0.0,1.0,2.0,0.0,0.0,12
7,1.0,20,20,Charlotte Lucas,6.0,0.0,0.0,2.0,0.0,2.0,0.0,0.0,10
8,1.0,21,21,Charlotte Lucas,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2
9,1.0,22,22,Charlotte Lucas,17.0,0.0,2.0,13.0,7.0,13.0,0.0,2.0,54


In [6]:
gold['Character'].value_counts()

Character
Elizabeth                      61
Jane                           53
Mr. Darcy                      49
Mrs. Bennet                    45
Mr. Bingley                    42
Mr. Bennet                     38
Wickham                        34
Lydia                          34
Charlotte Lucas                30
Mr. Collins                    29
Kitty                          26
Lady Catherine de Bourgh       26
Miss Bingley                   25
Miss Darcy                     24
Mrs. Gardiner                  19
Mr. Gardiner                   19
Sir William Lucas              18
Mary                           18
Mrs. Philips                   15
Miss de Bourgh                 15
Lady Lucas                     12
Mrs. Hurst                     11
Maria Lucas                    11
Colonel Forster                10
Colonel Fitzwilliam            10
Darcy's father                  7
Mr. Hurst                       6
Mr. Denny                       6
Mrs. Forster                    6
Miss

In [9]:
#see how many characters contained in character's by gold

char_set = set(gold['Character'].value_counts().index)
print(len(char_set), "unique characters")

55 unique characters


In [10]:
gold.head()

,Volume,Chapter,Graph Chapter,Character,N,DC,C,I,DN,A,B,FID,Score
0,1.0,7,7,Captain Carter,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,4
1,1.0,9,9,Captain Carter,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2
2,1.0,5,5,Charlotte Lucas,3.0,2.0,5.0,2.0,1.0,0.0,0.0,0.0,13
3,1.0,6,6,Charlotte Lucas,7.0,3.0,5.0,2.0,0.0,1.0,0.0,0.0,18
4,1.0,9,9,Charlotte Lucas,3.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,6


In [13]:
gold = gold.sort_values(["Graph Chapter", "Score"], ascending=[True, False])

In [15]:
gold.head(50)

,Volume,Chapter,Graph Chapter,Character,N,DC,C,I,DN,A,B,FID,Score
581,1.0,1,1,Mrs. Bennet,0.0,9.0,15.0,3.0,6.0,0.0,1.0,0.0,34
379,1.0,1,1,Mr. Bennet,6.0,13.0,3.0,4.0,1.0,0.0,1.0,0.0,28
417,1.0,1,1,Mr. Bingley,4.0,10.0,0.0,5.0,0.0,1.0,0.0,0.0,20
60,1.0,1,1,Elizabeth,2.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,5
667,1.0,1,1,Mrs. Long,2.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,5
126,1.0,1,1,Jane,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2
234,1.0,1,1,Lady Lucas,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2
246,1.0,1,1,Lydia,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2
572,1.0,1,1,Mr. Morris,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
692,1.0,1,1,Sir William Lucas,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
